# Dazo v0 — ProofWriter recurrent-depth pilot

This notebook is safe to rerun in the same Colab runtime. It updates `8dazo/dazo`, reuses existing ProofWriter data, trains the pilot when needed, and can recover an already-trained checkpoint even if an older notebook saved it under a nested `/content/dazo/...` path.

**Runtime:** GPU (Tesla T4 is enough for the pilot).


## 0. Setup / update repo safely


In [ ]:
!nvidia-smi || true

from pathlib import Path
import os, shutil, subprocess

# Record any pilot checkpoints that already exist before touching paths.
existing = list(Path('/content').glob('**/outputs/dazo-proofwriter-pilot/final/config.json'))
if existing:
    print('Existing pilot checkpoints:')
    for p in existing:
        print(' -', p.parent)

os.chdir('/content')
repo = Path('/content/dazo')
if (repo / '.git').exists():
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'reset', '--hard', 'origin/main'], check=True)
elif repo.exists():
    stale = Path('/content/dazo-stale')
    if stale.exists():
        shutil.rmtree(stale)
    repo.rename(stale)
    subprocess.run(['git', 'clone', '-q', 'https://github.com/8dazo/dazo.git', str(repo)], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/8dazo/dazo.git', str(repo)], check=True)

os.chdir(repo)
print('repo:', Path.cwd())
print('commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip())
!pip -q install -e ".[train]"


## 1. Architecture + loss invariants


In [ ]:
!pytest -q tests/test_core.py tests/test_losses.py


## 2. Prepare/reuse ProofWriter pilot

Training keeps only examples with gold query depth ≤3. Validation/test retain deeper cases. If the JSONL files already exist, this cell reuses them.


In [ ]:
from pathlib import Path
data_dir = Path('data/proofwriter-pilot')
needed = [data_dir/'train.jsonl', data_dir/'validation.jsonl', data_dir/'test.jsonl']
if all(p.exists() for p in needed):
    print('Reusing existing ProofWriter pilot data')
else:
    !python scripts/prepare_proofwriter.py \
      --output data/proofwriter-pilot \
      --train-max-depth 3 \
      --limit-train 3000 \
      --limit-eval 1000


## 3. Train the recurrent head (skip if you already have a completed pilot checkpoint)

If your previous run ended with `saved outputs/dazo-proofwriter-pilot/final`, you can skip this cell and go directly to section 4.


In [ ]:
!python train.py \
  --config configs/dazo-v0-small.json \
  --train data/proofwriter-pilot/train.jsonl \
  --eval data/proofwriter-pilot/validation.jsonl \
  --output outputs/dazo-proofwriter-pilot \
  --epochs 1 \
  --batch-size 4 \
  --grad-accum 2 \
  --lr 2e-4 \
  --depth-budgets 1,2,3,4,6,8


## 4. Recover checkpoint + test-time compute scaling

This cell searches the entire Colab runtime for the newest completed pilot checkpoint. That recovers checkpoints produced by earlier nested-path versions of this notebook.


In [ ]:
from pathlib import Path
import os, subprocess

candidates = list(Path('/content').glob('**/outputs/dazo-proofwriter-pilot/final/config.json'))
if not candidates:
    raise RuntimeError('No completed Dazo pilot checkpoint found. Run section 3 first.')

checkpoint = max((p.parent for p in candidates), key=lambda p: p.stat().st_mtime)
print('Using checkpoint:', checkpoint)

# Always evaluate with the latest source checkout, not a stale nested copy.
os.chdir('/content/dazo')
report = Path('/content/dazo/outputs/dazo-proofwriter-pilot/gate1-metrics.json')
report.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([
    'python', 'evaluate.py',
    '--model', str(checkpoint),
    '--data', 'data/proofwriter-pilot/test.jsonl',
    '--batch-size', '8',
    '--loops', '1,2,4,6,8',
    '--output', str(report),
], check=True)
print('Saved:', report)


## 5. Gate-1 summary


In [ ]:
import json
from pathlib import Path
r = json.loads(Path('outputs/dazo-proofwriter-pilot/gate1-metrics.json').read_text())
for loop, m in r['loops'].items():
    print(f"loops={loop:>2} accuracy={m['accuracy']:.4f} brier={m['brier']:.4f} ece={m['ece']:.4f} by_depth={m['by_depth']}")
print('overthinking_rate =', r.get('overthinking_rate'))
print('transitions =', json.dumps(r.get('correctness_transitions', {}), indent=2))


## 6. Next scale only if Gate 1 is informative

The 3k/1-epoch pilot is a pipeline and signal check. If recurrence shows a useful depth-dependent trend, scale to 20k–50k shallow-depth examples and 2–3 epochs before making any research claim.
